<a href="https://colab.research.google.com/github/quentin-garcia-3b1724235/canadian-banks-portfolio-analysis/blob/main/notebooks/canadian_banks_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Canadian Big Six Banks: Portfolio Performance and Risk Analysis

This notebook analyzes the performance and risk of a portfolio made of Canada’s six largest banks, using daily market data from Yahoo Finance.

##Section 0 : Setup

###0.1 Install and test yfinance

In [ ]:
!pip install --upgrade --quiet yfinance

In [ ]:
import yfinance as yf
import pandas as pd

print("yfinance version:", yf.__version__)

test = yf.download("RY.TO", period="5d", auto_adjust=False, progress=False)
test

yfinance version: 1.8.0


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,RY.TO,RY.TO,RY.TO,RY.TO,RY.TO,RY.TO
Date,,,,,,
2026-11-02,286.209991,286.209991,287.529999,284.000000,284.459991,2792000
2026-11-03,285.420013,285.420013,287.000000,283.529999,285.350006,2093600
2026-11-04,292.570007,292.570007,293.670013,287.500000,288.829987,2495100
2026-11-05,285.859985,285.859985,295.489990,285.510010,294.890015,3162000
2026-11-06,284.160004,284.160004,287.299988,283.250000,285.600006,2480300


###0.2 Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
PROJECT_DIR = "/content/drive/MyDrive/canadian-banks-portfolio"
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
print(os.listdir(PROJECT_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['data']


##Section 1 : Data

Download 20 years of daily prices for the six banks and the benchmark, then save them as CSV files.


###1.1 Parameters

In [ ]:
TICKERS = {
    "RY.TO": "RBC",
    "TD.TO": "TD",
    "BNS.TO": "Scotiabank",
    "BMO.TO": "BMO",
    "CM.TO": "CIBC",
    "NA.TO": "National Bank",
}
BENCHMARK = {"XIC.TO": "TSX Composite ETF"}

START_DATE = "2006-01-01"
END_DATE = "2026-01-01"

###1.2 Download daily prices

In [ ]:
all_tickers = list(TICKERS) + list(BENCHMARK)

raw = yf.download(
    all_tickers,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    actions=True,
    progress=False,
)

print(raw.shape)
raw.columns.get_level_values(0).unique()

(5020, 57)


Index(['Adj Close', 'Capital Gains', 'Close', 'Dividends', 'High', 'Low',
       'Open', 'Stock Splits', 'Volume'],
      dtype='object', name='Price')

###1.3 Extract adjusted prices, closing prices and dividends

In [ ]:
adj_close = raw["Adj Close"]
close = raw["Close"]
dividends = raw["Dividends"]

adj_close.tail()

Ticker,BMO.TO,BNS.TO,CM.TO,NA.TO,RY.TO,TD.TO,XIC.TO
Date,,,,,,,
2025-12-23,173.018265,97.054573,122.814125,170.928589,228.010574,125.145035,50.292862
2025-12-24,172.989197,97.313927,122.862648,171.532501,227.815720,125.378006,50.234104
2025-12-29,174.356415,97.285103,122.921371,171.885681,229.111420,125.795410,50.067642
2025-12-30,173.328552,97.333138,122.637558,170.374969,228.936035,126.008972,50.029243
2025-12-31,172.843735,97.237091,121.776329,169.325333,227.952118,125.572151,49.773232


###1.4 Check RBC’s recent dividends

In [ ]:
ry_dividends = yf.Ticker("RY.TO").dividends
ry_dividends.tail(6)

,Dividends
Date,
2025-07-24 09:30:00-04:00,1.54
2025-10-27 09:30:00-04:00,1.54
2026-01-26 09:30:00-05:00,1.64
2026-04-23 09:30:00-04:00,1.64
2026-07-27 09:30:00-04:00,1.76
2026-10-26 09:30:00-04:00,1.76


###1.5 Effect of dividends on RBC’s growth

In [ ]:
ry_close = close["RY.TO"].dropna()
ry_adj = adj_close["RY.TO"].dropna()

price_growth = ry_close.iloc[-1] / ry_close.iloc[0] - 1
total_growth = ry_adj.iloc[-1] / ry_adj.iloc[0] - 1

print(f"RBC price growth (Close):     {price_growth:.0%}")
print(f"RBC total growth (Adj Close): {total_growth:.0%}")

RBC price growth (Close):     410%
RBC total growth (Adj Close): 1006%


Over 2006–2025, RBC’s share price grew by 410%, but its total return with dividends reinvested reached 1,006%. More than half of the total gain therefore came from dividends. This is why all return calculations in this project use adjusted prices: closing prices alone would strongly understate the performance of dividend-paying stocks such as banks. Assumption: dividends are reinvested with no taxes or fees.

###1.6 Save the data to Google Drive

In [ ]:
adj_close.to_csv(f"{PROJECT_DIR}/data/adj_close_prices.csv")
close.to_csv(f"{PROJECT_DIR}/data/close_prices.csv")
dividends.to_csv(f"{PROJECT_DIR}/data/dividends.csv")

print(os.listdir(f"{PROJECT_DIR}/data"))

['adj_close_prices.csv', 'close_prices.csv', 'dividends.csv']


##2. Returns and data quality checks

Compute daily returns from adjusted prices and check that the data is reliable before any analysis.

###2.1 Load the saved data

In [ ]:
import numpy as np

prices = pd.read_csv(
    f"{PROJECT_DIR}/data/adj_close_prices.csv",
    index_col=0,
    parse_dates=True,
)
prices = prices[list(TICKERS) + list(BENCHMARK)]

print("Period:", prices.index.min().date(), "to", prices.index.max().date())
print("Trading days:", len(prices))
prices.head()

Period: 2006-01-03 to 2025-12-31
Trading days: 5020


,RY.TO,TD.TO,BNS.TO,BMO.TO,CM.TO,NA.TO,XIC.TO
Date,,,,,,,
2006-01-03,20.603638,14.001645,18.137629,26.424767,14.500978,13.001815,10.410629
2006-01-04,20.794586,14.053527,18.203585,26.627817,14.600212,12.911553,10.456909
2006-01-05,20.666538,14.053527,18.176418,26.667635,14.716300,12.920151,10.448230
2006-01-06,20.531765,13.985863,18.021242,26.516331,14.787442,12.834190,10.591413
2006-01-09,20.421684,14.035487,17.866049,26.337185,14.772465,12.834190,10.555261


###2.2 Check missing values

In [ ]:
print("Missing values per ticker:")
print(prices.isna().sum())

rows_with_missing = prices[prices.isna().any(axis=1)]
print("\nDays with at least one missing price:", len(rows_with_missing))
rows_with_missing

Missing values per ticker:
RY.TO     0
TD.TO     0
BNS.TO    0
BMO.TO    0
CM.TO     0
NA.TO     0
XIC.TO    0
dtype: int64

Days with at least one missing price: 0


,RY.TO,TD.TO,BNS.TO,BMO.TO,CM.TO,NA.TO,XIC.TO
Date,,,,,,,


In [ ]:
prices_clean = prices.dropna()
print("Days removed:", len(prices) - len(prices_clean))

Days removed: 0


###2.3 Compute simple and log returns

In [ ]:
simple_returns = prices_clean.pct_change().dropna()
log_returns = np.log(prices_clean / prices_clean.shift(1)).dropna()

simple_returns.head()

,RY.TO,TD.TO,BNS.TO,BMO.TO,CM.TO,NA.TO,XIC.TO
Date,,,,,,,
2006-01-04,0.009268,0.003705,0.003636,0.007684,0.006843,-0.006942,0.004445
2006-01-05,-0.006158,0.000000,-0.001492,0.001495,0.007951,0.000666,-0.000830
2006-01-06,-0.006521,-0.004815,-0.008537,-0.005674,0.004834,-0.006653,0.013704
2006-01-09,-0.005361,0.003548,-0.008612,-0.006756,-0.001013,0.000000,-0.003413
2006-01-10,0.008580,-0.005786,-0.001520,0.000755,0.004690,0.006363,-0.001782


###2.4 Consistency check with Section 1

In [ ]:
total_from_simple = (1 + simple_returns["RY.TO"]).prod() - 1
total_from_log = np.exp(log_returns["RY.TO"].sum()) - 1

print(f"RBC total return from simple returns: {total_from_simple:.0%}")
print(f"RBC total return from log returns:    {total_from_log:.0%}")

(simple_returns - log_returns).abs().max()

RBC total return from simple returns: 1006%
RBC total return from log returns:    1006%


,0
RY.TO,0.010103
TD.TO,0.014306
BNS.TO,0.012762
BMO.TO,0.015153
CM.TO,0.016604
NA.TO,0.016971
XIC.TO,0.007978


###2.5 Look for extreme and zero returns

In [27]:
big_moves = simple_returns[(simple_returns.abs() > 0.10).any(axis=1)]
print("Days with a move larger than 10%:", len(big_moves))
big_moves.round(3)

Days with a move larger than 10%: 16


,RY.TO,TD.TO,BNS.TO,BMO.TO,CM.TO,NA.TO,XIC.TO
Date,,,,,,,
2008-10-14,0.146,0.131,0.113,0.125,0.141,0.054,0.071
2008-11-20,-0.134,-0.127,-0.133,-0.123,-0.124,-0.167,-0.083
2008-11-24,0.069,0.039,0.038,0.053,0.037,0.127,0.035
2008-11-28,0.111,0.096,0.128,0.069,0.120,0.075,0.039
2008-12-19,0.007,0.025,-0.010,0.009,0.020,0.126,0.014
2009-02-24,0.103,0.075,0.084,0.061,0.078,0.058,0.024
2009-02-26,0.064,0.059,0.061,0.119,0.061,0.097,0.028
2009-03-10,0.144,0.110,0.123,0.102,0.107,0.062,0.038
2009-03-23,0.076,0.103,0.079,0.060,0.083,0.102,0.047


In [33]:
simple_returns.loc["2008-12-15":"2008-12-31", ["NA.TO", "XIC.TO"]].round(3)

,NA.TO,XIC.TO
Date,,
2008-12-15,-0.071,-0.004
2008-12-16,-0.021,0.029
2008-12-17,0.001,0.019
2008-12-18,-0.077,-0.054
2008-12-19,0.126,0.014
2008-12-22,0.036,-0.026
2008-12-23,0.010,0.001
2008-12-24,-0.004,-0.004
2008-12-29,0.055,0.042


In [28]:
(simple_returns == 0).sum()

,0
RY.TO,49
TD.TO,38
BNS.TO,51
BMO.TO,31
CM.TO,37
NA.TO,60
XIC.TO,124


In [31]:
(simple_returns["XIC.TO"] == 0).groupby(simple_returns.index.year).sum()

,XIC.TO
Date,
2006,3
2007,0
2008,1
2009,9
2010,12
2011,6
2012,12
2013,10
2014,14


###2.6 Check the benchmark’s capital gains distributions

In [32]:
xic_capital_gains = yf.Ticker("XIC.TO").capital_gains
xic_capital_gains

,Capital Gains
Date,


### 2.7 Interpretation

Only 16 trading days show a daily move larger than 10% for at least one ticker, and all of them fall within two crises: the 2008–2009 global financial crisis and the COVID-19 crash of March 2020. Most of these days (11 out of 16) are sharp rebounds rather than falls, which is typical of crisis periods. The largest bank-specific move (National Bank, December 2008) was checked against press articles and is linked to the bank’s exposure to the Canadian asset-backed commercial paper crisis. Days with no price change are rare for banks (about 1% of days) and more frequent for XIC (2.5%), mostly in calm years such as 2010–2014, with no sign of stale prices. One limitation: a trading halt on the Toronto Stock Exchange on December 17, 2008 distorts the split of returns between December 17 and 18, although the two-day total is consistent. Conclusion: the data is reliable, and extreme days are real market events, so they are kept, as they matter most for risk measures.